# GRPO Reward-Design Ablation — Kaggle Free-Tier Runner

Runs the **full scope** (Qwen2.5-1.5B, variants A0–A4, 5 seeds, GSM8K + MATH-500) on a
**free T4** using QLoRA (4-bit base + LoRA adapter).

**Before running — set these in the right panel:**
- Settings → Accelerator → **GPU T4 × 2**
- Settings → **Internet → On** (needs a one-time phone verification)

This sweep spans several sessions. See **Weekly loop** at the bottom — the driver
resumes automatically and never recomputes a finished job.


## 1. Get the code + install


In [ ]:
# Replace USERNAME with your GitHub user (private repo is fine if you add a token).
!git clone https://github.com/USERNAME/grpo-reward-ablation.git 2>/dev/null || echo 'already cloned'
%cd grpo-reward-ablation
!pip install -q -r requirements.txt
print('install done')

## 2. Prepare data (idempotent — safe to re-run)


In [ ]:
import os
if not os.path.isdir('data/gsm8k_train'):
    !python prepare_data.py --split train --out data/gsm8k_train
else:
    print('data/gsm8k_train already present')

## 3. Restore prior results

`/kaggle/working` is wiped between sessions, so each week you'll save your results as a
**Kaggle Dataset** and attach it next time (see Weekly loop). This cell pulls any prior
results back in so the driver knows what's already finished.


In [ ]:
import glob, shutil, os
os.makedirs('results', exist_ok=True)
restored = 0
# your attached results dataset mounts under /kaggle/input/<slug>/
for src in glob.glob('/kaggle/input/*/results/*.json') + glob.glob('/kaggle/input/*/*.json'):
    dst = os.path.join('results', os.path.basename(src))
    if not os.path.exists(dst):
        shutil.copy(src, dst); restored += 1
print(f'restored {restored} prior result file(s)')
!python runner.py status

## 4. Run pending jobs (background-safe)

Keep `--minutes` under your session limit (T4 sessions cap at ~9 h). To run unattended,
use **Save Version → Save & Run All (Commit)** — it executes in the background.

_Eval over the full test sets on a T4 is slow; if sessions run short, add
`--eval-limit 500 --n-samples 4` to the command to speed eval up without touching training._


In [ ]:
# QLoRA is on by default (required to fit 1.5B on 16 GB). Full scope otherwise.
!python runner.py run --minutes 480 --model Qwen/Qwen2.5-1.5B-Instruct --max-steps 500

## 5. Status + aggregate (works on partial results)


In [ ]:
!python runner.py status
print()
!python -m analysis.aggregate --results results --metric 'pass@1' --baseline A0 --dataset gsm8k
print()
!python -m analysis.aggregate --results results --metric 'pass@1' --baseline A0 --dataset math500

## Weekly loop

1. **Commit** (Save Version → Save & Run All). New results land in this version's
   `/kaggle/working/grpo-reward-ablation/results/` output.
2. Create or **update a Kaggle Dataset** from that `results/` folder
   (e.g. `grpo-ablation-results`).
3. Next session: **Add Input → your results dataset**, then Run All. `runner.py` skips
   finished jobs and continues from where it stopped.

Repeat until `status` shows **25/25 done**, then run the final aggregate for your tables.
